# ETL fundos

In [21]:
import pandas as pd
from pathlib import Path

In [28]:
caminho_fundos = Path("fundos")
df_fundos_def = pd.DataFrame()
for fundo in caminho_fundos.glob('*.csv'):
    df_fundo = pd.read_csv(fundo, encoding='latin1', sep=';', dtype={'TP_NEGOC': str})
    cols_importantes = ["DT_COMPTC", "CNPJ_FUNDO_CLASSE", "DENOM_SOCIAL","TP_APLIC", "CNPJ_FUNDO_CLASSE_COTA", "TP_ATIVO","NM_FUNDO_CLASSE_SUBCLASSE_COTA", "VL_MERC_POS_FINAL"]
    df_fundo = df_fundo[cols_importantes]
    df_fundo = df_fundo[df_fundo['VL_MERC_POS_FINAL'] >= 100000]
    df_fundos_def = pd.concat([df_fundos_def, df_fundo], axis=0, ignore_index=True)

df_fundos_def.shape

(187921, 8)

In [31]:
df_fundos_def['DT_COMPTC'].value_counts()

DT_COMPTC
2026-05-31    71343
2026-06-30    69870
2026-07-31    46708
Name: count, dtype: int64

# DF -> Grafo

In [32]:
import networkx as nx

In [35]:
df_fundos_def['DT_COMPTC'] = pd.to_datetime(df_fundos_def['DT_COMPTC'])

In [37]:
resultados_centralidade = []

"""
esse for irá "fatiar" o df pelos meses
a var mes vai guardar o "rótulo" do mês (ex: 05-2026)
dados_mes é o df em si
"""
for mes, dados_mes in df_fundos_def.groupby(df_fundos_def['DT_COMPTC'].dt.to_period('M')):

    # construção do grafo direcional
    G_mes = nx.from_pandas_edgelist(
        dados_mes,
        source='CNPJ_FUNDO_CLASSE',
        target="CNPJ_FUNDO_CLASSE_COTA",
        edge_attr="VL_MERC_POS_FINAL",
        create_using=nx.DiGraph()
    )

    # cálculo da centralidade de grau
    in_degree = nx.in_degree_centrality(G_mes)

    # armazenando os resultados
    for cnpj, score in in_degree.items():
        resultados_centralidade.append({
            "Mes" : str(mes),
            "CNPJ" : cnpj,
            "Central_Grau_Ent" : score
        })

In [38]:
# matriz de features
df_features = pd.DataFrame(resultados_centralidade)
df_features = df_features.sort_values(by=['CNPJ', 'Mes']).reset_index(drop=True)

In [39]:
df_features.head(10)

,Mes,CNPJ,Central_Grau_Ent
0,2026-05,00.068.305/0001-35,0.000000
1,2026-06,00.068.305/0001-35,0.000000
2,2026-07,00.068.305/0001-35,0.000000
3,2026-05,00.071.477/0001-68,0.000000
4,2026-06,00.071.477/0001-68,0.000000
5,2026-07,00.071.477/0001-68,0.000000
6,2026-05,00.083.181/0001-67,0.000097
7,2026-06,00.083.181/0001-67,0.000099
8,2026-07,00.083.181/0001-67,0.000136
9,2026-05,00.089.915/0001-15,0.000000
